# String and dates

In [0]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

emp.show()

In [0]:
# Case when

from pyspark.sql.functions import when, col
emp_gender_fixed = emp.withColumn("new_gender", when(col("gender") == "Male", "M")
                                  .when(col("gender") == "Female", "F")
                                  .otherwise(None))

emp_gender_fixed.show()

In [0]:
# Case with expr
from pyspark.sql.functions import expr
emp_gender_fixed_1 = emp.withColumn("new_gender", expr("case when gender = 'Male' then 'M' when gender = 'Female' then 'F' else null end"))
emp_gender_fixed_1.show()

In [0]:
# Regex replace
from pyspark.sql.functions import regexp_replace

emp_name_fixed = emp_gender_fixed_1.withColumn("new_name", regexp_replace(col("name"), "J", "Z"))
emp_name_fixed.show()  

In [0]:
# Date transformation
from pyspark.sql.functions import to_date

emp_date_fixed = emp_name_fixed.withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))
emp_date_fixed.show()

In [0]:
emp_date_fixed.printSchema()

In [0]:
# Add current date
from pyspark.sql.functions import current_date

emp_date_fixed = emp_date_fixed.withColumn("current_date", current_date())
emp_date_fixed.show()
# Add current timestamp
from pyspark.sql.functions import current_timestamp

emp_dated = emp_date_fixed.withColumn("current_timestamp", current_timestamp())
emp_dated.show()


In [0]:
# See full data

emp_dated.show(truncate=False)

In [0]:
# Drop null gender records
emp_1 = emp_dated.na.drop()
emp_1.show()

In [0]:
# Fix null values
from pyspark.sql.functions import coalesce, lit

emp_null_df = emp_dated.withColumn("new_gender", coalesce(col("new_gender"), lit("0")))
emp_null_df.show()


In [0]:
# Drop old columns and rename new columns

emp_final = emp_null_df.drop("gender", "name").withColumnRenamed("new_gender", "gender").withColumnRenamed("new_name", "name")
emp_final.show()

In [0]:
# Write data

emp_final.write.format("csv").save("/Volumes/workspace/pyspark/output/emp_final2.csv")

In [0]:
# COnvert date to string and extract date information
from pyspark.sql.functions import date_format

emp_fixed = emp_final.withColumn("date_string", date_format(col("hire_date"), "yyyy"))
emp_fixed.show()

In [0]:
# Extract zone
emp_fixed = emp_fixed.withColumn("zone", date_format(col("current_timestamp"), "z"))
emp_fixed.show()